# MICE補完の可視化と検証 (完全版)
このノートブックでは、以下の2点を確認します。
1. **金利データ (M1-M8)** の補完がカーブの形状を維持しているか
2. **外部指標 (USDJPY, JGB_Future等)** の欠損が、周囲のトレンドに馴染む形で補完されているか

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.processing import load_and_clean_data

# 1. データの読み込み (処理済みデータと生データ)
df = load_and_clean_data("../data/BOJ_data.xlsx", "../data/BOJ_meeting_history.csv")
raw_df = pd.read_excel("../data/BOJ_data.xlsx").iloc[1:]
raw_df["Date"] = pd.to_datetime(raw_df["日付"], format="%Y年%m月%d日")

# 生データのリネーム（比較用）
rename_map = {
    "JPBOJ1ONI=TRDT (MID_PRICE)": "M1", "JPY=": "USDJPY", 
    "JPY= (MID_PRICE)": "USDJPY", "JGBc1 (TRDPRC_1)": "JGB_Future",
    ".N225 (TRDPRC_1)": "Nikkei225", ".DXY (TRDPRC_1)": "DXY"
}
raw_df = raw_df.rename(columns=rename_map)
print(f"Data loaded. Rows: {len(df)}")

## 1. 補完箇所の可視化関数
実データを青点、補完データを赤のX印で表示する汎用的な関数を作成します。

In [ ]:
def plot_financial_data(processed_df, raw_df, col_name, days=365):
    # 表示期間の絞り込み
    end_date = processed_df["Date"].max()
    start_date = end_date - pd.Timedelta(days=days)
    
    p_df = processed_df[(processed_df["Date"] >= start_date) & (processed_df["Date"] <= end_date)].copy()
    r_df = raw_df[(raw_df["Date"] >= start_date) & (raw_df["Date"] <= end_date)].copy()
    
    # 補完フラグの特定（カラム名が異なる場合に対応）
    imputed_flag_col = f"{col_name}_is_imputed"
    if imputed_flag_col in p_df.columns:
        imputed_mask = p_df[imputed_flag_col] == 1
    else:
        # フラグがない場合は生データのNaNから特定
        imputed_mask = p_df["Date"].isin(r_df[r_df[col_name].isnull()]["Date"])
    
    plt.figure(figsize=(15, 5))
    # 折れ線（全体）
    plt.plot(p_df["Date"], p_df[col_name], color="gray", alpha=0.3, label="Interpolated Trend")
    # 実データ（青点）
    plt.scatter(p_df[~imputed_mask]["Date"], p_df[~imputed_mask][col_name], color="blue", s=15, label="Actual")
    # 補完データ（赤X）
    plt.scatter(p_df[imputed_mask]["Date"], p_df[imputed_mask][col_name], color="red", marker="x", s=60, label="Imputed (MICE)")
    
    plt.title(f"Imputation Visualization: {col_name}")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

# 金利データ(M1)の確認
plot_financial_data(df, raw_df, "M1")

## 2. 外部指標（USDJPY, JGB_Future）の補完確認
外部指標についても、金利データとの相関を利用して自然に補完されているか確認します。

In [ ]:
# USDJPYの確認
plot_financial_data(df, raw_df, "USDJPY")

# JGB先物の確認
plot_financial_data(df, raw_df, "JGB_Future")

## 3. イールドカーブの整合性
補完された日のM1-M8を繋いだとき、カーブが不自然に折れ曲がっていないか確認します。

In [ ]:
m_cols = [f"M{i}" for i in range(1, 9)]
imputed_days = df[df[[f"{c}_is_imputed" for c in m_cols]].sum(axis=1) > 0].sample(5, random_state=42)

plt.figure(figsize=(10, 6))
for _, row in imputed_days.iterrows():
    label = row["Date"].strftime("%Y-%m-%d")
    plt.plot(range(1, 9), row[m_cols], marker="o", label=label, alpha=0.7)
    # 補完箇所を強調
    for i, col in enumerate(m_cols):
        if row[f"{col}_is_imputed"] == 1:
            plt.scatter(i+1, row[col], color="red", marker="x", s=100, zorder=5)

plt.title("Yield Curve on Imputed Days (Red X = Imputed)")
plt.xticks(range(1, 9), m_cols)
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()